# Activity 3: Keyword Search Breaks, Embeddings Fix It, Then Keyword Search Wins Again

**Week 6 Day 4 | Real PDFs, real mess, and three retrieval algorithms that each fail differently**

**Estimated time:** 90 minutes
**Difficulty:** Intermediate
**Format:** Individual
**Prerequisites:** [Activity 0](./Activity_0_Environment_and_API_Setup.md) complete, `OPENAI_API_KEY` working

## The job

Activity 2 classified one message at a time. Real support and research work usually starts the other way around: a question comes in, and you have to find the *right* passage out of hundreds of pages before you can answer it. That is search, and it is the first half of every RAG system you will hear about for the rest of your career.

The corpus is not clean, hand-written sentences. It is three real NLP research papers in `pdfs/`, extracted straight from PDF, carrying every artifact real extraction produces: words broken across lines, typographic ligatures, glued-together tokens, no paragraph structure to speak of.

You will build three retrieval methods and watch each one fail:

1. **TF-IDF** ranks by rare-word overlap. It cannot see meaning.
2. **BM25** fixes TF-IDF's math. It still cannot see meaning.
3. **Embeddings** see meaning. They go blind on rare exact tokens, which is precisely what BM25 is best at.

That last pair is why production retrieval systems usually run both and combine them, and that is where this notebook ends.

## How this notebook works

Same shape as Activities 1 and 2. Every idea is explained, then worked once on something small enough to read in full, then handed to you as a `TODO` for the real thing. Run the worked cells and read the output before you write the `TODO` below them.

## What you will learn

- What `pypdf` actually returns, and why extraction quality is a retrieval bug
- What `chunk_size` and `overlap` really do, seen on text short enough to read whole
- TF-IDF computed by hand, one matrix at a time, before `scikit-learn` does it for you
- Why BM25 replaced TF-IDF in every serious search engine, and what its two knobs control
- What an embedding vector is and what a "high" similarity score actually means
- Reciprocal Rank Fusion: combining two rankings whose scores are not comparable

---
## Setup

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

---
# 1. What a PDF actually gives you

`pdfs/` holds nine real NLP papers. Start with one, and look hard at what comes out before trusting any of it.

A PDF does not store paragraphs. It stores instructions for placing glyphs at coordinates on a page. `pypdf` reads those instructions back and guesses where the words and lines were. Usually that guess is good. The places it is wrong are exactly the places your search will quietly break.

In [ ]:
from pypdf import PdfReader

reader = PdfReader("pdfs/1508.07909v5.pdf")
print(f"{len(reader.pages)} pages")

A `PdfReader` also carries the document's metadata, written by whatever tool produced the file. This is free provenance, and in a real pipeline you would store it alongside every chunk so you can answer "where did this text come from" later.

In [ ]:
for key, value in reader.metadata.items():
    print(f"{key:<20} {value}")

Note what is and is not there. There is a producer and a creation date, but the `/Title` is a LaTeX artifact rather than the paper's real title, and there is no author list you could trust. This is normal. PDF metadata is written by a typesetting tool, not by a librarian, so treat it as a hint, never as ground truth.

Now the text itself. `extract_text()` works one page at a time.

In [ ]:
page = reader.pages[0]
page_text = page.extract_text()

print(f"{len(page_text)} characters on page 1")
print()
print(page_text[:400])

Printed like that it looks fine. That is the trap. `print` renders the string the way a human reads it. Look at the same characters the way Python stores them, with `repr`, and the mess appears.

In [ ]:
print(repr(page_text[:400]))

Three things to find in that output:

1. **`\n` everywhere.** Those are line breaks from the printed page layout, not sentence or paragraph breaks. A line ends because the column ran out of width.
2. **Words split across lines**, like `vocabu-\nlary`. The hyphen is a typesetting hyphen, not part of the word.
3. **A ligature.** `ﬁxed` is not spelled with `f` then `i`.

That third one deserves its own cell, because it is the kind of bug that costs an afternoon.

In [ ]:
ligature = "ﬁxed"
plain = "fixed"

print(ligature, plain)
print("look identical:", ligature == plain)
print("lengths:", len(ligature), len(plain))
print("first character:", ascii(ligature[0]))

Two strings that render identically, are not equal, and do not even have the same length. `ﬁ` is a single character, `U+FB01`, that the typesetter substituted for the pair.

Keyword search is exact string matching underneath. A student searching `fixed vocabulary` will not match this page, and nothing will tell them why. Embeddings are more forgiving here, but they are not immune either.

Here is the same class of problem in a form that will matter later in this notebook. The paper reports a metric called CHRF3. Count how it survived extraction.

In [ ]:
full_text = " ".join(p.extract_text() for p in reader.pages)

print("CHRF3 :", full_text.count("CHRF3"))
print("CHR F3:", full_text.count("CHR F3"))

The token a keyword search needs does not exist in the extracted text. It was split by a stray space, so `CHRF3` scores exactly zero against all 189 chunks you are about to build, no matter how many times the paper discusses it.

Write this down, because it is the real lesson of this section: **extraction quality is a retrieval bug, not a formatting nuisance.** No amount of clever ranking recovers a token that extraction destroyed.

One more look before moving on. Page lengths vary a lot, which is worth knowing before you decide how to split the text.

In [ ]:
for i, p in enumerate(reader.pages):
    print(f"page {i}: {len(p.extract_text()):>5} characters")

Pages run from about 3,200 to just over 5,000 characters, and a page boundary falls wherever the column ran out of room, never where an idea starts or stops. Pages are not a useful unit of retrieval: too long to be specific, too uneven to compare, and cut in the wrong places. You need your own unit.

First, get the whole document as one string.

In [ ]:
def extract_text(path):
    reader = PdfReader(path)
    return " ".join(page.extract_text() for page in reader.pages)


paper_text = extract_text("pdfs/1508.07909v5.pdf")
print(f"{len(paper_text):,} characters")

---
# 2. Chunking, seen up close

You cannot search a 43,000 character string. You cannot embed it either: embedding models have an input limit, and even without one, a single vector for a whole paper would average everything the paper says into one blurry point. Ask it about subword units and it would answer about the general vibe of machine translation research.

So you split the text into **chunks**. Two parameters control that split, and both are usually explained in one sentence and then never made concrete. Let us make them concrete.

## 2.1 What `chunk_size` actually buys you

`chunk_size` is a number of characters. That number is the unit of everything downstream: it is what gets scored, what gets retrieved, and what gets pasted into a model's context window as evidence. So look at what different sizes actually contain.

In [ ]:
start = 6000

for size in (200, 800, 2000):
    print("=" * 70)
    print(f"chunk_size = {size}")
    print("=" * 70)
    print(paper_text[start:start + size])
    print()

Read all three before continuing.

- **200 characters** is a fragment. Precise if it happens to contain your answer, but hand it to a model as evidence and there is not enough there to reason from.
- **800 characters** is roughly a paragraph. Enough to state a complete idea, small enough that most of it is on topic.
- **2000 characters** spans several ideas. It will match many queries weakly, and rank high for none of them strongly, because the on-topic sentence is diluted by everything around it.

That dilution is the real cost of a large chunk, and it is not obvious until you have watched a ranking degrade. Small chunks are precise and context-poor. Large chunks are context-rich and imprecise. There is no correct answer, only a trade-off you should be able to defend.

## 2.2 What `overlap` actually prevents

Here is the part that is almost always hand-waved. Take a passage short enough to read in full.

In [ ]:
demo = (
    "Byte pair encoding merges the most frequent pair of symbols in the training corpus. "
    "This lets the model represent a rare word as a sequence of subword units. "
    "Unknown words therefore never need an UNK token at translation time."
)
print(len(demo), "characters")
print(demo)

Now split it into 80 character chunks with no overlap at all, the simplest thing that could work.

In [ ]:
naive_chunks = [demo[i:i + 80] for i in range(0, len(demo), 80)]

for i, c in enumerate(naive_chunks):
    print(f"[{i}] {c!r}")

Look at the boundary between chunk 1 and chunk 2. Chunk 1 ends with `Un` and chunk 2 begins with `known words`. The word `Unknown` was torn in half, and with it the phrase `Unknown words`. Neither chunk contains that phrase.

Prove it rather than trusting the eyeball.

In [ ]:
print("phrase is in the original text:", "Unknown words" in demo)
print("phrase is in ANY chunk        :", any("Unknown words" in c for c in naive_chunks))

`True`, then `False`. The information did not get corrupted, it got **cut**, and a search for `Unknown words` now returns nothing from a passage that plainly discusses unknown words.

The fix is to let each chunk start a little before the previous one ended, so anything sitting on a boundary lands whole inside at least one chunk. That backward step is the overlap.

In [ ]:
overlap = 30
stride = 80 - overlap

overlapped_chunks = [demo[i:i + 80] for i in range(0, len(demo), stride)]

for i, c in enumerate(overlapped_chunks):
    print(f"[{i}] {c!r}")

print()
print("phrase is in ANY chunk:", any("Unknown words" in c for c in overlapped_chunks))

`True`. Every chunk now begins 30 characters earlier than it otherwise would, and chunk 3 reaches back far enough to swallow the whole phrase.

That is the entire idea. **Overlap is insurance against a boundary landing in the middle of something that matters.** You are paying for that insurance with duplication: the overlapping text is stored, scored, and embedded more than once.

## 2.3 The arithmetic

The one line worth understanding is `stride = chunk_size - overlap`. The stride is how far the window moves each step, so it, not `chunk_size`, controls how many chunks you get.

In [ ]:
rows = []
for chunk_size, ov in [(800, 0), (800, 100), (800, 400), (300, 100), (2000, 100)]:
    stride = chunk_size - ov
    n_chunks = len(range(0, len(paper_text), stride))
    rows.append({
        "chunk_size": chunk_size,
        "overlap": ov,
        "stride": stride,
        "n_chunks": n_chunks,
        "duplication": f"{ov / chunk_size:.0%}",
    })

pd.DataFrame(rows)

Two things to take from that table. Going from 0 to 400 overlap at the same chunk size nearly doubles the chunk count, so it nearly doubles your embedding bill and your storage. And an overlap of 400 on an 800 character chunk means half of your corpus is stored twice, which is a lot to pay for boundary insurance.

An overlap of 10 to 15 percent of `chunk_size` is the usual default, which is where `800` and `100` come from.

Now write the chunker.

In [ ]:
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    stride = chunk_size - overlap
    for start in range(0, len(text), stride):
        chunks.append(text[start:start + chunk_size])
    return chunks


print(len(chunk_text(paper_text)), "chunks")

Now build the corpus you will search for the rest of this notebook: three papers, chunked, each chunk tagged with an id and its source file so a result is always traceable back to a document.

In [ ]:
PAPER_FILES = ["1508.07909v5.pdf", "1301.3781v3.pdf", "glove.pdf"]

CORPUS = []
for fname in PAPER_FILES:
    text = extract_text(f"pdfs/{fname}")
    for i, chunk in enumerate(chunk_text(text)):
        CORPUS.append({"id": f"{fname}#{i}", "source": fname, "text": chunk})

print(f"{len(CORPUS)} chunks from {len(PAPER_FILES)} papers")

In [ ]:
print(CORPUS[10]["id"])
print()
print(CORPUS[10]["text"])

That chunk starts mid-sentence and ends mid-sentence. `chunk_text` has no idea what a sentence is, it counts characters. That is the real cost of the simplest possible chunking strategy, and it is exactly why structure-aware splitters exist. You will meet one in Activity 4.

Also note: these papers are three different lengths, but every chunk is the same 800 characters. Remember that when you get to BM25, because it changes what one of its two knobs is worth.

---
# 3. TF-IDF, built by hand

Now the actual search. Before importing anything, decide what "this chunk matches that query" should even mean.

The naive answer is: count how many query words appear in the chunk. That fails immediately, because the highest-scoring chunk for almost any query would be whichever chunk says `the` most often. Common words are frequent and meaningless. Rare words are infrequent and informative.

TF-IDF is the smallest idea that fixes this, and it is two ideas multiplied:

- **TF (term frequency):** how often a term appears in this document. More is more relevant.
- **IDF (inverse document frequency):** how rare the term is across all documents. Rarer is more informative.

You are going to compute both by hand on a corpus small enough to hold in your head, then check your work against `scikit-learn`.

In [ ]:
TOY = [
    "the claim was denied because the policy had lapsed",
    "the claim was approved and paid within ten days",
    "the adjuster reviewed the claim and requested a police report",
    "the flood damage was excluded from this policy",
    "the policy covers flood damage after a waiting period",
]

for i, doc in enumerate(TOY):
    print(f"[{i}] {doc}")

Five short claim notes. Step one is tokenizing, which here just means lowercasing and splitting on non-letters.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


toy_tokens = [tokenize(doc) for doc in TOY]
print(toy_tokens[0])

vocabulary = sorted({t for doc in toy_tokens for t in doc})
print()
print(f"{len(vocabulary)} unique terms")
print(vocabulary)

## 3.1 The term-document matrix

Every document becomes a row, every vocabulary term a column, and each cell is a raw count. This matrix is the thing all of classical search is built on, and it is worth seeing once as an actual table.

In [ ]:
counts = pd.DataFrame(
    [[doc.count(term) for term in vocabulary] for doc in toy_tokens],
    columns=vocabulary,
)
counts

Mostly zeros, which is why real search engines store this sparsely, and why `scikit-learn` hands you a sparse matrix rather than a `DataFrame`.

Find the `the` column. It is the only column with a count above 1 in several rows, and it is the least useful word in the corpus. Raw counts are actively misleading.

## 3.2 Document frequency, then IDF

**Document frequency (`df`)** is how many documents a term appears in at all, regardless of how many times.

In [ ]:
N = len(TOY)
doc_freq = (counts > 0).sum(axis=0)

print(f"N = {N} documents")
print()
print(doc_freq.sort_values(ascending=False).head(8))

`the` appears in every document. `claim` and `policy` in three each. Now turn that into a weight, where rarer means larger. The textbook formula is `log(N / df)`.

In [ ]:
idf = np.log(N / doc_freq)

print(idf.sort_values().head(6))
print()
print(idf.sort_values(ascending=False).head(6))

Look at the bottom of the first list: **`the` has an IDF of exactly 0.0.**

That is the whole idea of IDF in one number. A term that appears in every document cannot possibly help you tell those documents apart, so its weight is zero and it drops out of the scoring entirely. Nobody had to write a stopword list. The math removed `the` because it earned nothing.

At the other end, terms appearing in one document out of five get `log(5/1)`, about 1.61, the highest weight available in this corpus.

## 3.3 TF times IDF

Multiply each row of raw counts by the IDF weights.

In [ ]:
tfidf = counts * idf
tfidf.round(2)

The `the` column is now entirely zero. Rare, specific terms like `flood`, `lapsed`, and `adjuster` carry the weight.

## 3.4 Scoring a query

A query is just another short document. Vectorize it the same way, then compare it to every row. The comparison is **cosine similarity**, which measures the angle between two vectors and ignores their length.

Length has to be ignored, or a long document would beat a short one purely for having more words in it. Cosine asks "do these point in the same direction", not "are these the same size".

In [ ]:
def cosine_similarity(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)


query = "flood damage policy"
query_counts = np.array([tokenize(query).count(term) for term in vocabulary])
query_vector = query_counts * idf.values

for i, doc in enumerate(TOY):
    score = cosine_similarity(query_vector, tfidf.iloc[i].values)
    print(f"{score:.3f}  [{i}] {doc}")

Documents 3 and 4, the two that actually discuss flood damage and policies, rank at the top. Document 0 mentions `policy` and scores something, but less. Documents 1 and 2 share only zero-weight words with the query and score 0.000.

You just built a search engine. It is about fifteen lines.

## 3.5 Checking against scikit-learn

`TfidfVectorizer` does all of the above. It also uses a slightly different IDF formula, and the difference is worth seeing so you are not confused later when its numbers do not match yours.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

check = TfidfVectorizer(norm=None, smooth_idf=False, token_pattern=r"[a-z0-9]+")
check.fit(TOY)

sk_idf = pd.Series(check.idf_, index=check.get_feature_names_out())
comparison = pd.DataFrame({"mine": idf, "sklearn": sk_idf})
comparison["difference"] = comparison["sklearn"] - comparison["mine"]
comparison.sort_values("mine").head(6).round(3)

Every difference is exactly 1.0. `scikit-learn` uses `log(N / df) + 1`.

The `+ 1` exists so that a term appearing in every document still contributes its raw count rather than vanishing completely. It is a deliberate, defensible choice, and it is different from the textbook formula you just implemented. `smooth_idf=True`, the default you would normally leave on, adds another adjustment that prevents a division by zero for unseen terms.

The lesson is not which formula is right. It is that **"TF-IDF" names a family of closely related formulas**, so when two systems disagree on scores, check their definitions before assuming one is broken. Rankings usually survive these differences. Absolute scores never do.

## 3.6 On the real corpus

Same technique, 189 messy chunks instead of 5 clean sentences.

In [ ]:
texts = [doc["text"] for doc in CORPUS]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(texts)

print("matrix shape:", tfidf_matrix.shape)
print(f"vocabulary:   {len(vectorizer.vocabulary_):,} unique terms")
print(f"density:      {tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.2%} non-zero")

189 rows, one per chunk, and several thousand columns, one per distinct term. Over 99 percent of that matrix is zeros.

Look at what the IDF weights picked out of real, messy text.

In [ ]:
real_idf = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())

print("LOWEST idf (in nearly every chunk, nearly worthless):")
print(real_idf.sort_values().head(8).round(2).to_string())
print()
print(f"HIGHEST idf is {real_idf.max():.2f}, and it is shared by "
      f"{(real_idf == real_idf.max()).sum():,} of {len(real_idf):,} terms")

The low-IDF list is exactly what you would expect from academic prose: `the`, `of`, `and`, plus words every NLP paper uses.

The high end is more interesting than a top-8 list would suggest. Nearly half the vocabulary appears in exactly one chunk, so nearly half the vocabulary is **tied** at the maximum weight. Sorting descending and printing the top few would just give you those ties in alphabetical order, which tells you nothing. That is a small but useful habit: when a sorted list looks alphabetical, you are probably looking at ties, not a ranking.

Sitting in that tied bucket is the extraction damage from section 1. Go find it.

In [ ]:
ligature_terms = [t for t in vectorizer.get_feature_names_out() if "ﬁ" in t or "ﬂ" in t]

print(f"{len(ligature_terms)} vocabulary terms contain a ligature")
print(ligature_terms[:12])

`classiﬁcation`, `artiﬁcial`, `beneﬁt`: these are ordinary words that the index has stored as terms no user will ever type. Someone searching `classification` gets zero matches from every chunk where the word was typeset with a ligature.

And because these mangled terms are rare by construction, IDF hands them near-maximum weight. **The noisiest tokens in your index are the ones the ranking function trusts most.** That is not a flaw in TF-IDF, it is TF-IDF working exactly as designed on input that section 1 already warned you about. In a production pipeline you would normalize this during extraction, well before any of the search code runs.

Now the search function.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity


def keyword_search(query, top_k=3):
    query_vector = vectorizer.transform([query])
    scores = sk_cosine_similarity(query_vector, tfidf_matrix)[0]
    ranked = sorted(zip(CORPUS, scores), key=lambda pair: pair[1], reverse=True)
    return ranked[:top_k]


def show(results):
    for doc, score in results:
        print(f"{score:.3f}  {doc['id']:<22} {doc['text'][:90].replace(chr(10), ' ')}...")


show(keyword_search("subword units for rare words"))

Strong match, right paper. That query borrows the paper's own vocabulary almost word for word, which is the situation TF-IDF handles well. Hold onto that, because section 5 asks the same question in different words.

---
# 4. BM25, or why nobody actually ships TF-IDF

If you open Elasticsearch, OpenSearch, Lucene, or the keyword half of almost any production hybrid retriever, you will not find TF-IDF. You will find **BM25**, a ranking function from the 1990s that is still the default today. It keeps IDF and replaces TF with something less naive.

TF-IDF makes two assumptions that are wrong, and BM25 fixes both.

## 4.1 Problem one: term frequency should not be linear

Under TF-IDF, a chunk mentioning `translation` 20 times scores 20 times higher than a chunk mentioning it once.

In [ ]:
term_idf = 1.5

for tf in [1, 2, 5, 20, 100]:
    print(f"tf={tf:>3}  tf-idf weight = {tf * term_idf:>6.2f}")

Ask yourself whether that is true. A chunk that says `translation` once is probably about something else and mentioned it. A chunk that says it five times is clearly about translation. Is a chunk that says it 100 times *twenty times more about translation* than the one that said it five times?

No. It is probably a table of contents, or a reference list. The first occurrence of a term is enormously informative, the second adds a lot, the tenth adds almost nothing. The relationship should **saturate**.

BM25 replaces the raw `tf` with this, where `k1` controls how fast saturation kicks in:

```
tf * (k1 + 1) / (tf + k1)
```

In [ ]:
def saturated_tf(tf, k1=1.5):
    return tf * (k1 + 1) / (tf + k1)


print(f"{'tf':>5} {'tf-idf (linear)':>16} {'bm25 (saturating)':>18}")
for tf in [1, 2, 3, 5, 10, 20, 100, 1000]:
    print(f"{tf:>5} {tf * 1.0:>16.2f} {saturated_tf(tf):>18.2f}")

Read the right-hand column top to bottom. Going from 1 occurrence to 2 is a large jump. Going from 20 to 100 is nearly nothing, and from 100 to 1000 is almost exactly nothing. The curve flattens toward a ceiling of `k1 + 1`, which is 2.5 at the default `k1=1.5`.

That ceiling is the point. **No single term can dominate a BM25 score through sheer repetition.** Keyword stuffing, whether it is a spammer's doing or just a reference list repeating an author name, stops working.

`k1` is the knob: lower saturates faster, higher behaves more like linear TF-IDF.

## 4.2 Problem two: document length

The second wrong assumption is subtler. A long document contains more words, so it is more likely to contain your query terms by chance alone. Cosine normalization in TF-IDF handles some of this, but bluntly.

BM25 compares each document's length to the corpus average, and the `b` knob controls how much that comparison matters.

```
tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / avgdl))
```

In [ ]:
def bm25_tf(tf, dl, avgdl, k1=1.5, b=0.75):
    return tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / avgdl))


avgdl = 800

print("Same term, appearing 3 times, in documents of different lengths:")
for dl in [200, 400, 800, 1600, 3200]:
    label = "average" if dl == avgdl else ("shorter" if dl < avgdl else "longer")
    print(f"  length {dl:>5} ({label:>7}): {bm25_tf(3, dl, avgdl):.3f}")

Three mentions in a 200 character document scores clearly higher than three mentions in a 3200 character one. In the short document, that term is a large fraction of everything the document says. In the long one, it might be incidental.

`b` runs from 0 (ignore length entirely) to 1 (normalize fully). The default 0.75 is a compromise that has survived thirty years of people trying to beat it.

**Now the honest caveat for this notebook.** Nearly every chunk in `CORPUS` is exactly 800 characters, because that is what `chunk_text` produces. When all documents are the same length, `dl / avgdl` is 1 for all of them, and `b` does nothing at all.

In [ ]:
lengths = pd.Series([len(doc["text"]) for doc in CORPUS])
print(lengths.describe().round(1).to_string())
print()
print("chunks shorter than 800 chars:", (lengths < 800).sum())

Only the final chunk of each paper is short. So on this corpus BM25's length normalization is close to a no-op, and essentially all of its advantage over TF-IDF comes from saturation.

That will not be true of your next corpus. The moment you retrieve over documents of genuinely different sizes, support tickets next to policy documents, chat messages next to PDFs, `b` starts doing real work.

## 4.3 BM25's IDF is different, and it barely matters

BM25 also swaps in its own IDF, derived from a probabilistic model of relevance rather than a plain log ratio. Compare the two at your actual corpus size, because the result is not what people usually assume.

In [ ]:
def bm25_idf(df, N):
    return np.log(1 + (N - df + 0.5) / (df + 0.5))


n_chunks = len(CORPUS)
rows = []
for df_val in [1, 5, 20, 95, 180, n_chunks]:
    rows.append({
        "df": df_val,
        "in_chunks": f"{df_val}/{n_chunks}",
        "tfidf_idf": round(float(np.log(n_chunks / df_val)), 3),
        "bm25_idf": round(float(bm25_idf(df_val, n_chunks)), 3),
    })

pd.DataFrame(rows)

The two curves nearly coincide. BM25 is slightly *lower* for the rarest terms and slightly higher for the most common ones, and through the middle they are indistinguishable.

One genuine difference: BM25's IDF never reaches exactly zero, so a term appearing in every single chunk still contributes a sliver, where TF-IDF zeroes it out completely.

The useful conclusion here is a negative one. **BM25 does not beat TF-IDF because of IDF.** Whatever advantage it has comes from saturation and length normalization, sections 4.1 and 4.2. When you read that a system "upgraded from TF-IDF to BM25", that is what actually changed.

## 4.4 BM25 on the real corpus

You have seen every piece. In practice you use the library, because it handles the bookkeeping and because `rank_bm25` is what you would reach for at work.

One thing the library does not do is tokenize. You have to hand it tokens, which means **your tokenizer is now part of your ranking function.** Reuse the same one you wrote for the toy corpus so the query and the corpus are split identically.

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [tokenize(t) for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

print(f"indexed {len(tokenized_corpus)} chunks")
print(f"average chunk length: {bm25.avgdl:.1f} tokens")
print(f"k1={bm25.k1}  b={bm25.b}")

In [ ]:
def bm25_search(query, top_k=3):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    ranked = sorted(zip(CORPUS, scores), key=lambda pair: pair[1], reverse=True)
    return ranked[:top_k]


show(bm25_search("subword units for rare words"))

Right paper again, on a completely different score scale, and the chunk ranking overlaps TF-IDF's without matching it. On a query phrased in the paper's own vocabulary both methods work, so this is not where BM25 earns its reputation.

Where it earns it is section 7. First, the failure they share.

---
# 5. The failure both keyword methods share

Everything so far has asked questions in the papers' own words. Real users do not do that. They do not know your documents' vocabulary, which is exactly why they are searching.

Ask the same underlying question the way somebody who has never read the paper would ask it.

In [ ]:
paraphrase = "what makes it possible to translate a word the model has never encountered"

print("--- TF-IDF ---")
show(keyword_search(paraphrase))
print()
print("--- BM25 ---")
show(bm25_search(paraphrase))

Look at the **source paper** in those ids, not the scores.

The question describes the exact problem that `1508.07909v5.pdf` was written to solve. Both methods rank chunks from `1301.3781v3.pdf` (word2vec) or `glove.pdf` first. This is not a near miss with low confidence. It is the wrong document, ranked first, with a perfectly ordinary-looking score.

And critically: **BM25 fails the same way TF-IDF does.** Saturation and length normalization made the scoring smarter. They did nothing about the actual problem, because the actual problem is not arithmetic.

Diagnose it directly. Compare the query's tokens against the tokens of a chunk that genuinely answers it.

In [ ]:
right_answer = next(d for d in CORPUS if d["id"] == "1508.07909v5.pdf#2")

query_tokens = set(tokenize(paraphrase))
chunk_tokens = set(tokenize(right_answer["text"]))

print("query tokens:", sorted(query_tokens))
print()
print("shared with the correct chunk:", sorted(query_tokens & chunk_tokens))

The overlap is almost entirely function words: the kind of terms IDF already decided are worthless. The words carrying the question's meaning, `never` and `encountered`, do not appear in the chunk at all. The chunk expresses the same idea as `open-vocabulary` and `rare` instead, and neighbouring chunks use `unseen` and `unknown`.

A human reads `a word the model has never encountered` and `out-of-vocabulary word` as the same thing. Every method in this notebook so far is, structurally, incapable of that. They compare tokens. Nothing in TF-IDF or BM25 knows that two different strings can mean the same thing, and no amount of tuning `k1` or `b` will teach them.

That is the wall. Getting past it needs a fundamentally different representation.

---
# 6. Embeddings

An **embedding** is a list of numbers representing a piece of text, produced by a model trained so that text with similar meaning gets similar numbers. The words themselves are gone. What survives is position in a space where distance means something.

Start with one sentence and look at what actually comes back.

In [ ]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="the claim was denied because the policy had lapsed",
)

vector = response.data[0].embedding

print(f"type:       {type(vector).__name__}")
print(f"dimensions: {len(vector)}")
print(f"first 8:    {[round(v, 4) for v in vector[:8]]}")
print(f"norm:       {np.linalg.norm(vector):.4f}")

1536 floating point numbers. None of them is individually interpretable: there is no "insurance dimension" you could go read. The meaning is in the whole vector's position, not any one component.

The norm is 1.0, meaning OpenAI returns unit-length vectors. Convenient detail: for unit vectors, cosine similarity and dot product are the same computation.

## 6.1 Calibrating what a score means

Before trusting any similarity number, find out what high and low look like on text you already understand.

In [ ]:
def embed(text):
    return client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding


anchor = "the claim was denied because the policy had lapsed"

candidates = [
    "the claim was rejected since the coverage had expired",
    "we turned down the request because they had not paid",
    "flood damage is excluded from this policy",
    "the restaurant serves excellent pasta on weekends",
]

anchor_vector = embed(anchor)

print(f"anchor: {anchor}")
print()
for c in candidates:
    print(f"{cosine_similarity(anchor_vector, embed(c)):.3f}  {c}")

Read those four numbers carefully, because they are the whole point.

The first candidate shares almost no vocabulary with the anchor. `denied` versus `rejected`, `policy` versus `coverage`, `lapsed` versus `expired`. TF-IDF would score it near zero, because they have barely a content word in common. The embedding model scores it far above everything else, because it means the same thing. That single number is the entire reason this section exists.

At the other end, the sentence about pasta scores near zero, which is the model behaving sensibly.

Now look at the two in the middle, because they are the ones that should change how you use these scores. One is a full restatement of the anchor in plain language. The other is a different claim about a different part of the policy. They mean quite different things, and they score within a few hundredths of each other.

So there is no threshold that cleanly separates "means the same thing" from "is about the same topic". Any cutoff you pick between those two numbers is arbitrary, and it will move the moment you change corpus, model, or query phrasing. This is why retrieval **ranks** candidates against each other instead of thresholding on an absolute score, and why "is 0.47 a good similarity?" is a question with no answer.

## 6.2 Embedding the corpus

189 chunks, one at a time, would be 189 round trips. The endpoint accepts a list, so send batches. Default to this pattern any time you embed more than a handful of items.

In [ ]:
def embed_batch(texts, batch_size=100):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model="text-embedding-3-small", input=batch)
        all_embeddings.extend(item.embedding for item in response.data)
    return all_embeddings


corpus_embeddings = embed_batch(texts)
print(f"{len(corpus_embeddings)} embeddings, {len(corpus_embeddings[0])} dimensions each")

189 chunks in two API calls rather than 189. At this model's pricing the whole corpus costs a fraction of a cent, which is worth knowing before you assume embedding is the expensive part of RAG. It is not. Generation is.

In [ ]:
def semantic_search(query, top_k=3):
    query_vector = embed(query)
    scored = [(doc, cosine_similarity(query_vector, v)) for doc, v in zip(CORPUS, corpus_embeddings)]
    return sorted(scored, key=lambda pair: pair[1], reverse=True)[:top_k]


show(semantic_search(paraphrase))

The right paper, `1508.07909v5.pdf`, on a query that shares almost no literal vocabulary with it.

Nothing in this notebook told the model that `never encountered` and `out-of-vocabulary` mean the same thing. It learned that from training on an enormous amount of text, and that knowledge is what the 1536 numbers encode. This is the same underlying capability you used in Activity 2 for sentiment, pointed at retrieval instead.

---
# 7. Where embeddings lose badly

Here is where most tutorials stop, having concluded that embeddings are simply better. They are not. They are better at one thing and clearly worse at another, and if you ship a retrieval system without knowing which is which, users will find the gap for you.

The BPE paper compares itself against a segmentation tool called **Morfessor**. Search for it by name, the way a researcher actually would.

In [ ]:
print("--- BM25 ---")
show(bm25_search("Morfessor"))
print()
print("--- Embeddings ---")
show(semantic_search("Morfessor"))

BM25 returns chunks that discuss Morfessor. The embedding model returns chunks from the paper's **bibliography**.

Confirm which chunks genuinely contain the word.

In [ ]:
for doc in CORPUS:
    if "Morfessor" in doc["text"]:
        print(doc["id"])

Compare that list against the two result sets above. BM25 found them. The embedding model did not, and it did not return something merely mediocre, it returned reference-list chunks: author names, venues, page numbers.

The reason is worth understanding, because it generalizes. `Morfessor` is a rare proper noun. The embedding model has essentially no learned meaning for it, so it falls apart into subword pieces that carry no useful signal, and the query vector ends up encoding something like "an unusual technical-looking proper noun". The nearest chunks to *that* are the ones densest in unusual proper nouns, which is precisely the bibliography.

This failure mode has a name, **vocabulary mismatch in reverse**, and the class of queries it hits is exactly the class real users care about most:

| Query type | Example | Winner |
| :--- | :--- | :--- |
| Rare proper noun | `Morfessor`, a vendor name, a drug name | Keyword |
| Identifier or code | policy number `ACD-4471`, error `E_TIMEOUT` | Keyword |
| Exact metric or number | `chrF3`, `60 000 merge operations` | Keyword |
| Natural language question | `why was my claim denied` | Embeddings |
| Paraphrase or synonym | `never encountered` for `out-of-vocabulary` | Embeddings |
| Cross-lingual or informal | slang, typos, another language | Embeddings |

A claims search that cannot find a policy number is broken, no matter how good its semantic understanding is. So do not choose. Use both.

---
# 8. Hybrid retrieval with Reciprocal Rank Fusion

You have two rankers that fail on opposite query types. The obvious move is to combine them. The obvious *implementation* of that move does not work.

In [ ]:
bm25_top = bm25_search("Morfessor", top_k=1)[0]
semantic_top = semantic_search("Morfessor", top_k=1)[0]

print(f"BM25 top score:      {bm25_top[1]:.3f}")
print(f"Embedding top score: {semantic_top[1]:.3f}")

You cannot add those. BM25 scores are unbounded, corpus-dependent, and query-dependent: a two-word query and a ten-word query produce completely different score magnitudes. Cosine similarities are bounded and, as section 6.1 showed, compressed into a narrow band. Adding them means the BM25 score decides everything. Normalizing them per query is possible but fragile, since the min and max shift with every query.

**Reciprocal Rank Fusion** sidesteps the problem entirely by throwing the scores away and keeping only the ranks. A document's contribution from each ranker is:

```
1 / (k + rank)
```

with `rank` starting at 1 and `k` a constant, conventionally 60. Sum that across rankers.

Two properties make this work. Ranks are comparable across any two systems, since position 1 means the same thing everywhere. And the `k` in the denominator flattens the curve so first place is not overwhelmingly more valuable than third, which means a document both rankers like at position 2 or 3 can beat a document only one ranker put first. That is exactly the behavior you want from a consensus.

In [ ]:
for rank in range(1, 11):
    print(f"rank {rank:>2}: {1 / (60 + rank):.5f}")

Now build it. Take a deeper slice from each ranker than you intend to return, so a document ranked 8th by one method and 2nd by the other can still surface.

In [ ]:
def hybrid_search(query, top_k=3, k=60, depth=20):
    fused = {}
    docs_by_id = {}
    for results in (bm25_search(query, depth), semantic_search(query, depth)):
        for rank, (doc, _score) in enumerate(results, start=1):
            docs_by_id[doc["id"]] = doc
            fused[doc["id"]] = fused.get(doc["id"], 0) + 1 / (k + rank)
    ranked = sorted(fused.items(), key=lambda pair: pair[1], reverse=True)
    return [(docs_by_id[doc_id], score) for doc_id, score in ranked[:top_k]]


print("--- Morfessor (BM25 wins alone) ---")
show(hybrid_search("Morfessor"))
print()
print("--- paraphrase (embeddings win alone) ---")
show(hybrid_search(paraphrase))

One function, and the top result is right for both queries: the rare-token query that embeddings botched, and the paraphrase that both keyword methods botched. Neither method alone does that.

**Be honest about the cost.** Fusion buys robustness at the top of the list and pays for it further down, because half of each ranker's mistakes get mixed in. Positions 2 and 3 are usually noisier than the winning method's positions 2 and 3 would have been. For RAG, where you typically hand a model the top 3 to 5 chunks and it can ignore an irrelevant one, that is a good trade. For a UI showing a user ten results, it is worth measuring rather than assuming.

Which is the honest end of this notebook: you now have three retrievers and a fusion of them, and **no evidence about which is best for your actual users**, because every conclusion here rests on two hand-picked queries. Activity 6 fixes that with a labeled evaluation set.

---
# 9. Scorecard

| | TF-IDF | BM25 | Embeddings | Hybrid (RRF) |
| :--- | :--- | :--- | :--- | :--- |
| Query in the document's own words | Good | Good | Good | Good |
| Natural-language paraphrase | Fails | Fails | Good | Good |
| Rare proper noun or identifier | Good | Good | Fails | Good |
| Cost to index | Free | Free | An API call per batch | Both |
| Cost per query | Free | Free | One API call | One API call |
| Explainable ranking | Yes | Yes | No | Partly |
| Handles typos and synonyms | No | No | Yes | Yes |
| New document added | Refit the vectorizer | Reindex | Embed and append | Both |

Two things to carry out of this table.

**Keyword search is not legacy technology.** BM25 is free, instant, needs no API, ranks explainably, and beats embeddings outright on the exact-match queries users issue constantly. Every serious retrieval system in production still runs it.

**"Which is better" is the wrong question.** They fail on disjoint query types. The engineering question is how to combine them and how to know whether the combination helped, which is Activity 6's job.

---
# Your Turn

Work in your own copy under `student-work/week6/day4/`.

1. **Add a fourth paper.** Put `"1810.04805v2.pdf"` (BERT) into `PAPER_FILES` and rebuild `CORPUS`, `tfidf_matrix`, `bm25`, and `corpus_embeddings`. Every one of those is stale the moment the corpus changes, and forgetting one is the single most common bug in this kind of code. Then ask a question about BERT in your own words and confirm `semantic_search` finds it.

2. **Break the tokenizer on purpose.** In `bm25_search`, replace `tokenize(query)` with `query.split()` and rerun `bm25_search("Morfessor")` and `bm25_search("BLEU")`. Explain in a markdown cell exactly why one still works and the other does not.

3. **Find a query where a zero score hides a failure.** Search for a term that appears nowhere in the corpus (`Zipf` is one). Look at what `keyword_search` returns and at the scores. What is it actually ranking, and what should a real system do when every score is 0?

4. **Sweep `chunk_size`.** Rebuild the corpus at `chunk_size=300` and again at `2000`, keeping overlap at 100, and rerun the paraphrase query through `semantic_search`. Do the retrieved chunks get more precise, or does the loss of surrounding context make them harder to actually use as evidence? Write down which you would ship and why.

**Stretch goal:** `hybrid_search` weights both rankers equally. Add a `weights=(1.0, 1.0)` parameter that scales each ranker's contribution, then find a query where shifting the weight toward BM25 improves the result and one where it makes it worse. You have just discovered the parameter every hybrid retrieval system has to tune, and the reason Activity 6 exists.

## What you did

- Pulled text out of a real PDF and found the ligature, the split token, and the page-length variance that make extraction a retrieval problem rather than a formatting one.
- Built chunking from the ground up, watching a phrase get severed at a boundary and then rescued by overlap.
- Computed TF-IDF by hand, one matrix at a time, and watched `the` fall to a weight of exactly zero without a stopword list.
- Replaced it with BM25, and saw specifically what saturation and length normalization each contribute, including the fact that length normalization does nothing on fixed-size chunks.
- Watched both keyword methods confidently return the wrong paper for a paraphrase, and diagnosed it as vocabulary mismatch rather than a scoring bug.
- Fixed that with embeddings, then found the query type where embeddings lose to BM25 badly enough to return a bibliography.
- Combined both with Reciprocal Rank Fusion, and got one ranker that handles both query types.

**Next:** [Activity 4](./Activity_4_RAG_from_Scratch_to_Chroma.ipynb) turns this retrieval step into a full RAG pipeline over these same papers, then swaps the by-hand search for Chroma and then for LlamaIndex, where the rank fusion you just wrote turns up as a single named component.